In [1]:
import numpy as np
import pandas as pd

In [2]:
from scipy.io import loadmat

long_mat = loadmat("corr-offsite-results_1-12_winter_hour_train60_multi1_th50.mat")
print(long_mat.keys())   
short_mat = loadmat("short_term_cov.mat")
print(short_mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'allError', 'allError_NWP', 'allNWPinfo', 'allVariance', 'allY_Fore', 'allY_NWP', 'allY_Test', 'allY_int', 'maes', 'maes_nwp', 'train_nwp'])
dict_keys(['__header__', '__version__', '__globals__', 'allVariance_Fore', 'allY_Fore', 'maes'])


In [3]:
long_mat['allVariance'].shape

(216, 216, 183)

In [13]:
import numpy as np

def rmse_over_horizons(y_true, y_pred, h_start, h_end):
    """
    """
    sl = slice(h_start-1, h_end)  
    err = y_pred[45:183,:, sl] - y_true[45:183,:, sl]
    return np.sqrt(np.mean(err**2))

rmse_1_6   = rmse_over_horizons(true[45:183,:,:], mu_avg[45:183,:,:], 1, 6)
rmse_7_24  = rmse_over_horizons(true[45:183,:,:], mu_avg[45:183,:,:], 7, 24)
rmse_1_24  = rmse_over_horizons(true[45:183,:,:], mu_avg[45:183,:,:], 1, 24)

print(rmse_1_6, rmse_7_24, rmse_1_24)


1.8585134744587737 2.193334365664712 2.114605119966982


In [14]:
import numpy as np

def extract_std_183_2_24(allVariance, loc_list=[8, 9], block_size=24):

    n_rolls = allVariance.shape[2]
    n_locs = len(loc_list)

    std_array = np.zeros((n_rolls, n_locs, block_size))

    for j, loc in enumerate(loc_list):
        start = (loc - 1) * block_size
        end = loc * block_size

        cov_sub = allVariance[start:end, start:end, :]   # (24, 24, 183)
        std_array[:, j, :] = np.sqrt(np.diagonal(cov_sub, axis1=0, axis2=1))

    return std_array
std_long = extract_std_183_2_24(long_mat['allVariance'], loc_list=[8, 9])
std_short = extract_std_183_2_24(short_mat['allVariance_Fore'], loc_list=[8, 9])

In [15]:
var_ens = 0.25 * (std_long ** 2) + 0.25 * (std_short ** 2)
std_avg = np.sqrt(var_ens)

Simple average code in the following chunck

In [16]:
mu_long = np.asarray(long_mat["allY_Fore"])[:, [7,8], :] 
mu_long = np.transpose(mu_long, (2, 1, 0))
mu_short = np.asarray(short_mat["allY_Fore"])[:, [7,8], :]
mu_short = np.transpose(mu_short, (2, 1, 0))

In [17]:
mu_long.shape

(183, 2, 24)

In [18]:
mu_avg = 0.5 * mu_long + 0.5 * mu_short

In [19]:
mu_ens = mu_avg[45:,:,:]

In [20]:
std_ens = std_avg[45:,:,:]

In [21]:
true = long_mat['allY_Test']
true = np.transpose(true, (2, 1, 0))
true[45:183,:,:].shape

(138, 2, 24)

In [36]:
## another benchmark ENS-S
import numpy as np

cal_window = 45

n_days, n_sites, horizon = mu_short.shape

# 138 test days
n_rolls = n_days - cal_window

# Store switching point selected for each roll
switch_points = np.zeros(n_rolls, dtype=int)

# Store final switched forecast
mu_switch = np.zeros((n_rolls, n_sites, horizon))
sigma_switch = np.zeros((n_rolls, n_sites, horizon))
# Store calibration RMSE for every candidate switching point
cal_rmse_switch = np.zeros((n_rolls, horizon + 1))


for r, t in enumerate(range(cal_window, n_days)):

    # -----------------------------------
    # 1. Previous 45-day calibration data
    # -----------------------------------
    short_cal = mu_short[t-cal_window:t]   # (45, 2, 24)
    long_cal  = mu_long[t-cal_window:t]    # (45, 2, 24)
    true_cal  = true[t-cal_window:t]       # (45, 2, 24)

    # -----------------------------------
    # 2. Try all possible switch points
    # -----------------------------------
    # s = 0  -> all LTM
    # s = 6  -> STM for hours 1-6, LTM for 7-24
    # s = 24 -> all STM

    for s in range(horizon + 1):

        pred_cal = np.empty_like(short_cal)
        pred_cal[:, :, :s] = short_cal[:, :, :s]
        pred_cal[:, :, s:] = long_cal[:, :, s:]
        rmse = np.sqrt(np.mean((pred_cal - true_cal) ** 2))
        cal_rmse_switch[r, s] = rmse

    best_s = np.argmin(cal_rmse_switch[r])

    switch_points[r] = best_s

    mu_switch[r, :, :best_s] = mu_short[t, :, :best_s]
    mu_switch[r, :, best_s:] = mu_long[t, :, best_s:]
    sigma_switch[r, :, :s] = std_short[t, :, :s]
    sigma_switch[r, :, s:] = std_long[t, :, s:]

rmse_1_6   = rmse_over_horizons(true[45:183,:,:], mu_switch[:,:,:], 1, 6)
rmse_7_24  = rmse_over_horizons(true[45:183,:,:], mu_switch[:,:,:], 7, 24)
rmse_1_24  = rmse_over_horizons(true[45:183,:,:], mu_switch[:,:,:], 1, 24)

print(rmse_1_6, rmse_7_24, rmse_1_24)

1.853309988160572 2.2047357731302957 2.1223417137785012


In [23]:
switch_points
sp = np.asarray(switch_points).flatten()

In [24]:
np.mean(switch_points)

5.173913043478261

In [25]:
np.quantile(switch_points,[0.1,0.25,0.3,0.4,0.5,0.6,0.75,0.8,0.9])

array([1., 1., 2., 2., 6., 7., 8., 8., 8.])

In [41]:
from scipy.io import savemat

savemat("ensemble_swichpoint.mat", {
    "mu": mu_switch,
    "sigma": sigma_switch,
    "y_true": true[45:,:,:]   
})

In [26]:
true.shape

(183, 2, 24)

In [27]:
short_mat["allVariance_Fore"].shape

(216, 216, 183)

In [33]:
from scipy.io import savemat

savemat("ensemble_simple_avg.mat", {
    "mu": mu_ens,
    "sigma": std_ens,
    "y_true": true[45:,:,:]   
})


In [28]:
import numpy as np

loc_idx = [7, 8] ## index for E05/E06
L_total = 9      ##total location for base model
ORDER = 'loc_first'   # flatten order

mu_long = np.asarray(long_mat["allY_Fore"])[:, loc_idx, :] ## pick data for E05/E06
mu_long = np.transpose(mu_long, (2, 1, 0))     ## 183*2*24

mu_short = np.asarray(short_mat["allY_Fore"])[:, loc_idx, :]
mu_short = np.transpose(mu_short, (2, 1, 0))        

y_raw = np.asarray(long_mat["allY_Test"]) ## 24*2*183
y_true = np.transpose(y_raw[:, :, :], (2, 1, 0)) 

D = mu_long.shape[0]  ## 183
L = 2

## flatten the data
if ORDER == 'hour_first':
    idx0 = np.array([h * L_total + loc_idx[0] for h in range(24)], dtype=int)
    idx1 = np.array([h * L_total + loc_idx[1] for h in range(24)], dtype=int)
elif ORDER == 'loc_first':
    idx0 = np.array([loc_idx[0] * 24 + h for h in range(24)], dtype=int)  ## 7*24+1-8*24 index for E05
    idx1 = np.array([loc_idx[1] * 24 + h for h in range(24)], dtype=int) ## index for E06
else:
    raise ValueError("ORDER must be 'hour_first' or 'loc_first'.")

# covariance for long term model
Sig_long_raw = np.asarray(long_mat["allVariance"])   # 216*216*183
if Sig_long_raw.ndim == 3 and Sig_long_raw.shape[0] == 24 * L_total:
    # (216,216,183) get covariance matrix by location
    Sig_long = np.empty((D, L, 24, 24))
    for d in range(D):
        S = Sig_long_raw[:, :, d] 
        Sig_long[d, 0] = S[np.ix_(idx0, idx0)]
        Sig_long[d, 1] = S[np.ix_(idx1, idx1)]
elif Sig_long_raw.ndim == 4 and Sig_long_raw.shape[:2] == (24, 24):
    # if raw covariance matrix in other form
    Ssel = Sig_long_raw[:, :, loc_idx, :]
    Sig_long = np.transpose(Ssel, (3, 2, 0, 1))
else:
    raise ValueError(f"Unexpected long covariance shape: {Sig_long_raw.shape}")

# covaraince for the short-term
Sig_short_raw = np.asarray(short_mat["allVariance_Fore"])  
if Sig_short_raw.ndim == 3 and Sig_short_raw.shape[0] == 24 * L_total:
    Sig_short = np.empty((D, L, 24, 24))
    for d in range(D):
        S = Sig_short_raw[:, :, d]
        Sig_short[d, 0] = S[np.ix_(idx0, idx0)]
        Sig_short[d, 1] = S[np.ix_(idx1, idx1)]
elif Sig_short_raw.ndim == 4 and Sig_short_raw.shape[:2] == (24, 24):
    if Sig_short_raw.shape[2] >= max(loc_idx)+1:
        Ssel = Sig_short_raw[:, :, loc_idx, :]
        Sig_short = np.transpose(Ssel, (3, 2, 0, 1))
    else:
        Sig_short = np.transpose(Sig_short_raw, (2, 0, 1))[:, None, :, :].repeat(2, axis=1)
elif Sig_short_raw.ndim == 2 and Sig_short_raw.shape == (24, 24):
    Sig_short = np.tile(Sig_short_raw, (D, 2, 1, 1))
else:
    raise ValueError(f"Unexpected short covariance shape: {Sig_short_raw.shape}")

# final mu and covaraince for the base model
muX, SigXX = mu_short, Sig_short     # (183,2,24), (183,2,24,24)
muY, SigYY = mu_long,  Sig_long      # (183,2,24), (183,2,24,24)

print("muX:", muX.shape, "SigXX:", SigXX.shape)
print("muY:", muY.shape, "SigYY:", SigYY.shape)
print("y_true:", y_true.shape)
# muX:   (183, 2, 24)  SigXX: (183, 2, 24, 24)
# muY:   (183, 2, 24)  SigYY: (183, 2, 24, 24)
# y_true:(183, 2, 24)


muX: (183, 2, 24) SigXX: (183, 2, 24, 24)
muY: (183, 2, 24) SigYY: (183, 2, 24, 24)
y_true: (183, 2, 24)


In [29]:
import numpy as np
from scipy.optimize import minimize
import time
# muX, muY, y_true: (D, 2, 24)
# SigXX, SigYY:     (D, 2, 24, 24)
D, L, H = muX.shape
assert (L, H) == (2, 24)

Wwin    = 45          # calibration window
S       = 1000        # Monte carlo
M_pairs = 10000       # pairs
seed    = 42          
I24     = np.eye(24)
eff     = D - Wwin    # test days

muZ_out      = np.empty((24, 2, eff))
SigZ_per_loc = np.empty((2, 24, 24, eff))
error_out    = np.empty((24, 2, eff))
alpha_seq    = np.empty(eff); ell_seq = np.empty(eff); rho_seq = np.empty(eff)
MAE_daily    = np.empty(eff); ES_daily = np.empty(eff)

t = np.arange(24, dtype=float)
TT_i = t[:, None]; TT_j = t[None, :]
rng_global = np.random.default_rng(seed)

def softplus(x):  
    return np.log1p(np.exp(x))

alpha_lo, alpha_hi = 0.05, 0.12 ## bound for alpha, change based on prior or unconstrained
bounds = [(alpha_lo, alpha_hi), (None, None), (None, None)]  # [alpha, l, r]

# warm start
theta_prev = np.array([0.10, 2.0, 0.0], dtype=float)  # [alpha0, l0, r0]
roll_times = np.empty(eff)
for d in range(Wwin, D):
    roll_start = time.perf_counter()
    k   = d - Wwin
    win = np.arange(d - Wwin, d)
    Wlen= len(win)

    eps_bank = rng_global.standard_normal((Wlen, S, 48))  # 48=2*24
    i_pair   = rng_global.integers(0, S, size=M_pairs)
    j_pair   = rng_global.integers(0, S, size=M_pairs)
    m_pairs  = (i_pair != j_pair)
    i_pair, j_pair = i_pair[m_pairs], j_pair[m_pairs]

    # Cholesky decomposition
    SX_list = [[None]*2 for _ in range(Wlen)]
    SY_list = [[None]*2 for _ in range(Wlen)]
    for kk, dd in enumerate(win):
        for loc in range(2):
            # SigXX chol
            A = SigXX[dd, loc]
            jitter = 1e-6
            for _try in range(8):
                try:
                    SX_list[kk][loc] = np.linalg.cholesky(A + jitter * I24)
                    break
                except np.linalg.LinAlgError:
                    jitter *= 10.0
            # SigYY chol
            B = SigYY[dd, loc]
            jitter = 1e-6
            for _try in range(8):
                try:
                    SY_list[kk][loc] = np.linalg.cholesky(B + jitter * I24)
                    break
                except np.linalg.LinAlgError:
                    jitter *= 10.0

    # objective of ES
    def es_objective(theta):
        # theta = [alpha, l, r]
        alpha, l, r = theta
        ell = softplus(l)     # >0
        rho = np.tanh(r)      # (-1,1)
        ## weight
        w = np.exp(-alpha * (np.arange(24) + 1.0))
        Wmat = np.diag(w); IminusW = np.diag(1.0 - w)

        # RBF K(ell) normalize
        dist2 = (TT_i - TT_j)**2
        K = np.exp(-0.5 * dist2 / (ell**2 + 1e-12))
        evals = np.linalg.eigvalsh((K + K.T)/2.0)
        Kt = K / max(evals.max(), 1e-12)
        C = rho * Kt  # 24x24

        total = 0.0
        for kk, dd in enumerate(win):
            muZ_loc, SigZ_loc = [], []

            for loc in range(2):
                # mu
                muZ = Wmat @ muX[dd, loc] + IminusW @ muY[dd, loc]
                # cross-covariance
                SigXY = SX_list[kk][loc] @ C @ SY_list[kk][loc].T
                # final covariance
                SigZ = (Wmat @ SigXX[dd, loc] @ Wmat
                        + IminusW @ SigYY[dd, loc] @ IminusW
                        + Wmat @ SigXY @ IminusW
                        + IminusW @ SigXY.T @ Wmat)

                muZ_loc.append(muZ)
                SigZ_loc.append(SigZ)

            
            jitter = 1e-6
            for _ in range(8):
                try:
                    L0 = np.linalg.cholesky(SigZ_loc[0] + jitter * I24); break
                except np.linalg.LinAlgError: jitter *= 10.0
            jitter = 1e-6
            for _ in range(8):
                try:
                    L1 = np.linalg.cholesky(SigZ_loc[1] + jitter * I24); break
                except np.linalg.LinAlgError: jitter *= 10.0

            Lbd = np.zeros((48,48))
            Lbd[0:24,0:24] = L0
            Lbd[24:48,24:48] = L1

            mu_flat = np.concatenate([muZ_loc[0], muZ_loc[1]], axis=0)
            y_flat  = np.concatenate([y_true[dd,0],  y_true[dd,1]],  axis=0)

            # Monte Carlo ES
            X = mu_flat[None,:] + eps_bank[kk] @ Lbd.T
            term1 = np.mean(np.linalg.norm(X - y_flat[None,:], axis=1))
            diffs = X[i_pair] - X[j_pair]
            term2 = 0.5 * np.mean(np.linalg.norm(diffs, axis=1))
            total += (term1 - term2)

        return total / Wlen

    # optimize L-BFGS-B
    opt = minimize(es_objective, x0=theta_prev, method="L-BFGS-B",
                   bounds=bounds, options=dict(maxiter=200, ftol=1e-9))
    ## optimized params
    alpha_hat, l_hat, r_hat = opt.x
    ell_hat   = softplus(l_hat)
    rho_hat   = np.tanh(r_hat)

    alpha_seq[k], ell_seq[k], rho_seq[k] = alpha_hat, ell_hat, rho_hat
    theta_prev = opt.x  # warm-start

    w = np.exp(-alpha_hat * (np.arange(24) + 1.0))
    Wmat = np.diag(w); IminusW = np.diag(1.0 - w)
    dist2 = (TT_i - TT_j)**2
    K = np.exp(-0.5 * dist2 / (ell_hat**2 + 1e-12))
    evals = np.linalg.eigvalsh((K + K.T)/2.0)
    Kt = K / max(evals.max(), 1e-12)
    C = rho_hat * Kt
    ## ensemble at the next day
    muZ_loc, SigZ_loc, errs = [], [], []
    for loc in range(2):
        jitter = 1e-6
        for _ in range(8):
            try:
                SX = np.linalg.cholesky(SigXX[d, loc] + jitter * I24); break
            except np.linalg.LinAlgError: jitter *= 10.0
        jitter = 1e-6
        for _ in range(8):
            try:
                SY = np.linalg.cholesky(SigYY[d, loc] + jitter * I24); break
            except np.linalg.LinAlgError: jitter *= 10.0

        muZ = Wmat @ muX[d, loc] + IminusW @ muY[d, loc]
        SigXY = SX @ C @ SY.T
        SigZ  = (Wmat @ SigXX[d, loc] @ Wmat
                 + IminusW @ SigYY[d, loc] @ IminusW
                 + Wmat @ SigXY @ IminusW
                 + IminusW @ SigXY.T @ Wmat)

        muZ_out[:, loc, k]         = muZ
        SigZ_per_loc[loc, :, :, k] = SigZ
        e = y_true[d, loc] - muZ
        error_out[:, loc, k]       = e

        muZ_loc.append(muZ); SigZ_loc.append(SigZ); errs.append(e)
    ## record MAE per day
    MAE_daily[k] = float(np.mean(np.abs(np.stack(errs, axis=0))))


    rng_eval = np.random.default_rng(seed + 1000 + k)
    eps_eval = rng_eval.standard_normal((S, 48))
    mu_flat = np.concatenate([muZ_loc[0], muZ_loc[1]], axis=0)
    y_flat  = np.concatenate([y_true[d,0],  y_true[d,1]],  axis=0)

    jitter = 1e-6
    for _ in range(8):
        try:
            L0 = np.linalg.cholesky(SigZ_loc[0] + jitter * I24); break
        except np.linalg.LinAlgError: jitter *= 10.0
    jitter = 1e-6
    for _ in range(8):
        try:
            L1 = np.linalg.cholesky(SigZ_loc[1] + jitter * I24); break
        except np.linalg.LinAlgError: jitter *= 10.0
    Lbd = np.zeros((48,48)); Lbd[0:24,0:24] = L0; Lbd[24:48,24:48] = L1

    X = mu_flat[None,:] + eps_eval @ Lbd.T
    term1 = np.mean(np.linalg.norm(X - y_flat[None,:], axis=1))
    i_eval = rng_eval.integers(0, S, size=M_pairs)
    j_eval = rng_eval.integers(0, S, size=M_pairs)
    m_eval = (i_eval != j_eval)
    diffs = X[i_eval[m_eval]] - X[j_eval[m_eval]]
    term2 = 0.5 * np.mean(np.linalg.norm(diffs, axis=1))
    ES_daily[k] = float(term1 - term2)
    
    roll_end = time.perf_counter()
    roll_times[k] = roll_end - roll_start
    ## monitor training phase
    if (k+1) % 10 == 0 or k == eff-1:
        print(f"[{k+1}/{eff}] alpha={alpha_hat:.4f}, ell={ell_hat:.3f}, rho={rho_hat:.3f}, "
              f"MAE={MAE_daily[k]:.4f}, ES={ES_daily[k]:.4f}, "
            f"roll time={roll_times[k]:.2f} s")

#print("Overall MAE:", float(np.mean(MAE_daily)), "Overall ES:", float(np.mean(ES_daily)))


def daily_rmse_both(err, sl): return np.sqrt((err[sl,:,:]**2).mean(axis=(0,1)))
def daily_mae_both(err, sl):  return np.abs(err[sl,:,:]).mean(axis=(0,1))

hrs_1_6, hrs_7_24, hrs_1_24 = slice(0,6), slice(6,24), slice(0,24)

rmse_1_6  = daily_rmse_both(error_out, hrs_1_6)
rmse_7_24 = daily_rmse_both(error_out, hrs_7_24)
rmse_1_24 = daily_rmse_both(error_out, hrs_1_24)

mae_1_6   = daily_mae_both(error_out, hrs_1_6)
mae_7_24  = daily_mae_both(error_out, hrs_7_24)
mae_1_24  = daily_mae_both(error_out, hrs_1_24)

rmse_1_6_overall  = float(np.sqrt((error_out[hrs_1_6,:,:]**2).mean()))
rmse_7_24_overall = float(np.sqrt((error_out[hrs_7_24,:,:]**2).mean()))
rmse_1_24_overall = float(np.sqrt((error_out[hrs_1_24,:,:]**2).mean()))
mae_1_6_overall   = float(np.abs(error_out[hrs_1_6,:,:]).mean())
mae_7_24_overall  = float(np.abs(error_out[hrs_7_24,:,:]).mean())
mae_1_24_overall  = float(np.abs(error_out[hrs_1_24,:,:]).mean())

print(f"Overall pooled RMSE (both locs): 1-6={rmse_1_6_overall:.4f}, 7-24={rmse_7_24_overall:.4f}, 1-24={rmse_1_24_overall:.4f}")
print(f"Overall pooled MAE  (both locs): 1-6={mae_1_6_overall:.4f}, 7-24={mae_7_24_overall:.4f}, 1-24={mae_1_24_overall:.4f}")
## epf env could run smoothly

[10/138] alpha=0.1200, ell=0.876, rho=1.000, MAE=1.5164, ES=8.9216, roll time=2.01 s
[20/138] alpha=0.1200, ell=0.828, rho=1.000, MAE=1.1484, ES=7.8326, roll time=1.29 s
[30/138] alpha=0.1200, ell=0.458, rho=1.000, MAE=1.9820, ES=11.1980, roll time=5.51 s
[40/138] alpha=0.0950, ell=0.452, rho=1.000, MAE=1.5608, ES=9.1178, roll time=6.77 s
[50/138] alpha=0.0884, ell=0.213, rho=1.000, MAE=1.1030, ES=7.7726, roll time=6.12 s
[60/138] alpha=0.0875, ell=0.213, rho=1.000, MAE=1.4717, ES=9.2226, roll time=4.86 s
[70/138] alpha=0.1165, ell=0.213, rho=1.000, MAE=1.2766, ES=8.7041, roll time=3.46 s
[80/138] alpha=0.0963, ell=0.659, rho=1.000, MAE=1.6972, ES=10.8007, roll time=6.90 s
[90/138] alpha=0.1200, ell=0.700, rho=1.000, MAE=1.7823, ES=10.5878, roll time=3.43 s
[100/138] alpha=0.0963, ell=1.091, rho=1.000, MAE=1.2959, ES=7.9172, roll time=11.24 s
[110/138] alpha=0.0678, ell=1.512, rho=1.000, MAE=1.0968, ES=7.0798, roll time=9.24 s
[120/138] alpha=0.0722, ell=1.100, rho=1.000, MAE=1.6361, E

In [31]:
print("\nTiming summary")
print(f"Total time:       {roll_times.sum():.2f} s")
print(f"Mean per roll:    {roll_times.mean():.2f} s")
print(f"Median per roll:  {np.median(roll_times):.2f} s")
print(f"Std per roll:     {roll_times.std(ddof=1):.2f} s")
print(f"Min per roll:     {roll_times.min():.2f} s")
print(f"Max per roll:     {roll_times.max():.2f} s")


Timing summary
Total time:       1003.70 s
Mean per roll:    7.27 s
Median per roll:  5.42 s
Std per roll:     4.87 s
Min per roll:     1.29 s
Max per roll:     25.67 s


In [145]:
from scipy.io import savemat

## save the file
savemat("ensemble_es_constrain.mat", {
    "mu": muZ_out,
    "Sigma": SigZ_per_loc,
    "Error": error_out,
    "alpha": alpha_seq,
    "ell": ell_seq,
    "rho": rho_seq,
    "MAE_daily": MAE_daily,
    "ES_daily": ES_daily,
    "y_true": y_true   
})
